# P118 — Traducción automática neuronal de palabras raras con unidades de subpalabra

## 1. Título y paper

**Paper:** *Neural Machine Translation of Rare Words with Subword Units*  
**Autoría:** Rico Sennrich, Barry Haddow, Alexandra Birch  
**Año y venue:** 2016 · ACL 2016, 1715–1725  
**Nivel:** L2 · **Motor:** `bpe`  
**Ficha completa:** [`P118_bpe`](../../papers/foundational/P118_bpe/README.md)

**Hito:** Elimina el problema de la palabra desconocida haciendo que la unidad de vocabulario sea más pequeña que la palabra, con un algoritmo que la frecuencia decide sola.

- [doi:10.18653/v1/P16-1162](https://doi.org/10.18653/v1/P16-1162)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un vocabulario de palabras completas siempre se queda corto: llega una palabra que no estaba y el modelo solo puede emitir un símbolo de desconocido, aunque sus raíces y sufijos sí estuvieran en el entrenamiento.
2. Ejecutar una implementación mínima de la propuesta: Adaptar la compresión por pares de bytes: partir de caracteres y fusionar repetidamente el par de símbolos más frecuente, un número fijo de veces. El vocabulario resultante cubre cualquier cadena porque el peor caso es deletrear.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P05
- P06


## 4. Intuición

Si la unidad de vocabulario es la palabra, siempre llegará una que no estaba. Si es más pequeña que la palabra, el peor caso es deletrear — y deletrear siempre se puede.


## 5. Concepto mínimo

```text
BPE: empezar por caracteres y repetir k veces:
     fusionar el par de símbolos MÁS FRECUENTE del corpus

vocabulario = alfabeto ∪ {símbolos producidos por las k fusiones}
     → cobertura total, porque el alfabeto está dentro
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('bpe', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué porcentaje de palabras nuevas será desconocido con vocabulario de palabras?
2. ¿Cuántos trozos quedarán fuera de vocabulario con BPE?
3. ¿Qué se paga a cambio?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('bpe', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('bpe', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con vocabulario de palabras, **17 de 120** palabras de prueba son desconocidas (14,2 %) aunque sus raíces y sufijos sí se vieron. Con 60 fusiones, el vocabulario BPE tiene **87 unidades** —menos que las 100 palabras— y los trozos fuera de vocabulario son **0**. Se paga en longitud: **2,02 piezas** por palabra en vez de 1.


## 10. Comentario pedagógico

Mira las primeras fusiones: son parejas de letras frecuentes, no morfemas. Las últimas ya son sufijos completos. Nadie programó qué es un sufijo — la frecuencia lo descubre, y por eso el método funciona igual en idiomas cuya morfología el ingeniero no conoce.


## 11. Error o anti-patrón deliberado

Anti-patrón: agrandar el vocabulario de palabras hasta que «ya casi no haya desconocidas».


In [ ]:
print('Duplicar el vocabulario de palabras reduce las desconocidas, nunca las elimina.')
print('Y cada palabra nueva del vocabulario es una fila mas en la matriz de embeddings.')
print('BPE resuelve el problema en vez de posponerlo.')

## 12. Corrección

La perilla real es el número de fusiones:


In [ ]:
r = run_paper_lab('bpe', seed=3)['result']
print('vocabulario de palabras:', r['vocabulario_de_palabras'])
print('vocabulario BPE:', r['vocabulario_bpe'])
print('fuera de vocabulario:', r['trozos_fuera_de_vocabulario'])
print('piezas por palabra:', r['piezas_por_palabra'])

## 13. Desafío guiado

Explica el compromiso entre tamaño de vocabulario y longitud de secuencia, y por qué el coste de atención lo vuelve una decisión de arquitectura y no de preprocesado.


In [ ]:
r = run_paper_lab('bpe', seed=3)['result']
show(r)

## 14. Desafío autónomo

Tokeniza un corpus tuyo con dos tamaños de vocabulario y mide piezas por palabra en cada uno. Comprueba qué le pasa a los términos técnicos de tu dominio.


## 15. Evidencia de aprendizaje

Guarda las dos mediciones y el tamaño de vocabulario que elegirías, con el motivo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P118_bpe/README.md) · evaluación formal: [`assessments/papers/P118_bpe.md`](../../assessments/papers/P118_bpe.md)


## 16. Cierre

Abre la ruta de percepción: cómo entra el mundo en el modelo cuando no llega como texto limpio.


## 17. Conexión con el siguiente hito

- P123

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
